In [3]:
import pandas as pd
import sys
import os
import importlib


# === 1) FORCE THE PATH ===
# -----------
"""I first need to force the code to ignore any previous projects python looks for first"""
# -----------
# 1. Get the path to the current project folder
current_project_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
# 2. Force the path to the front of search lift
if current_project_path not in sys.path:
    sys.path.insert(0, current_project_path)
# 3. Once  Jupyter imports a file, it keeps it in its 'memory', even if I change the file on the hard drive.
# importlib.reload() allows me to clear jupyters memory, reads the config.py file, and ensures it views the correct path
from src import config
importlib.reload(config)

print("---------------------------------------------")
print(f"Correct Project Root: {config.PROJ_ROOT}")
print(f"Config Data Path:     {config.DATA_RAW}")
print("------------------------------------------------")

# 2) --- VERIFY THE PATHS ---
target_path = "/Users/joelangstaff/Downloads" # expected downloads path
big_file_path = config.DATA_RAW / "charts.csv"

# 3) === MINIMUM VIABLE SLICE ===
"""I need to create a smaller dataset as the 3GB will take up too much space in RAM
I can model the time series forecasting around a much smaller dataset (40mb) by taking Ed Sheeran"""

if str(config.DATA_RAW) == target_path and big_file_path.exists():
    print(f"\n Found charts.csv! Starting extraction...")

    target_artist = "Ed Sheeran"
    filtered_rows = []
    chunk_size = 1_000_000

    # --- RUN THE FILTER ---
    try:
        # I use the default engine (safer than pyarrow for chunks)
        with pd.read_csv(big_file_path, chunksize=chunk_size) as reader:
            for i, chunk in enumerate(reader):
                # Filter this chunk
                matches = chunk[chunk['artist'] == target_artist]

                # If we found matches, keep them
                if len(matches) > 0:
                    filtered_rows.append(matches)
                    print(f"   Chunk {i+1}: Found {len(matches)} rows...")

        # --- SAVE THE RESULT ---
        if filtered_rows:
            df_small = pd.concat(filtered_rows)

            # Define where to save inside the project
            save_path = config.PROJ_ROOT / "data" / "raw" / "ed_sheeran_charts.csv"

            # THE FIX: Force-create the folder structure if it's missing
            print(f"Ensuring directory exists: {save_path.parent}")
            save_path.parent.mkdir(parents=True, exist_ok=True)

            # Save file
            df_small.to_csv(save_path, index=False)
            print(f"\nSUCCESS! Saved {len(df_small)} rows.")
            print(f"File located at: {save_path}")
            print("Now delete 'charts.csv' from the Downloads folder.")

        else:
             print(f"Warning: No rows found for {target_artist}. Check spelling?")

    except Exception as e:
        print(f"Error: {e}")

else:
    print(f"\n Error: Could not find 'charts.csv' in {config.DATA_RAW}")
    print("Please ensure to unzip the file in downloads folder")

Config loaded. Pointing to raw data at: /Users/joelangstaff/Downloads
---------------------------------------------
Correct Project Root: /Users/joelangstaff/Library/Mobile Documents/com~apple~CloudDocs/Code/machine-learning/TOPICS/Time-Series Forecasting/Projects/spotify-forecasting
Config Data Path:     /Users/joelangstaff/Downloads
------------------------------------------------

 Found charts.csv! Starting extraction...
   Chunk 1: Found 19379 rows...
   Chunk 2: Found 23666 rows...
   Chunk 3: Found 17444 rows...
   Chunk 4: Found 15001 rows...
   Chunk 5: Found 13589 rows...
   Chunk 6: Found 14737 rows...
   Chunk 7: Found 22304 rows...
   Chunk 8: Found 23476 rows...
   Chunk 9: Found 19486 rows...
   Chunk 10: Found 22239 rows...
   Chunk 11: Found 30665 rows...
   Chunk 12: Found 32232 rows...
   Chunk 13: Found 15013 rows...
   Chunk 14: Found 2897 rows...
   Chunk 15: Found 6469 rows...
   Chunk 16: Found 6834 rows...
   Chunk 17: Found 6201 rows...
   Chunk 18: Found 6344